In [15]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import keras_tuner as kt


In [27]:
import pandas as pd
import numpy as np

In [17]:
data = pd.read_csv("/content/Real_Combine.csv")

In [18]:
data.head()

,T,TM,Tm,SLP,H,VV,V,VM,PM 2.5
0,7.4,9.8,4.8,1017.6,93.0,0.5,4.3,9.4,219.720833
1,7.8,12.7,4.4,1018.5,87.0,0.6,4.4,11.1,182.187500
2,6.7,13.4,2.4,1019.4,82.0,0.6,4.8,11.1,154.037500
3,8.6,15.5,3.3,1018.7,72.0,0.8,8.1,20.6,223.208333
4,12.4,20.9,4.4,1017.3,61.0,1.3,8.7,22.2,200.645833


In [19]:
# Separate features and target
X = data.iloc[:, :-1].copy()
y = data.iloc[:, -1].copy()

# Make sure all values are numeric; invalid values become NaN
X = X.apply(pd.to_numeric, errors="coerce")
y = pd.to_numeric(y, errors="coerce")

# Replace +/- infinity with NaN
X = X.replace([np.inf, -np.inf], np.nan)
y = y.replace([np.inf, -np.inf], np.nan)

# Remove rows containing missing/invalid values in either X or y
valid_rows = X.notna().all(axis=1) & y.notna()
X = X.loc[valid_rows].reset_index(drop=True)
y = y.loc[valid_rows].reset_index(drop=True)

print("Cleaned dataset shape:", X.shape)
print("Remaining NaNs in X:", X.isna().sum().sum())
print("Remaining NaNs in y:", y.isna().sum())


# Hyperparameters

How many Hidden Layers should we have ?

How many number of neurons we should have in hidden layers ?

Learning Rate

In [39]:
def build(hp):
    model = keras.Sequential()
    model.add(keras.Input(shape=(X_train.shape[1],)))

    num_layers = hp.Int("num_layers", min_value=1, max_value=6, step=1)

    for i in range(num_layers):
        model.add(
            layers.Dense(
                units=hp.Int(
                    f"units_{i}",
                    min_value=32,
                    max_value=256,
                    step=32
                ),
                activation="relu"
            )
        )

    model.add(layers.Dense(1, activation="linear"))

    learning_rate = hp.Choice(
        "learning_rate",
        values=[1e-2, 1e-3, 1e-4]
    )

    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=learning_rate),
        loss="mean_absolute_error",
        metrics=[keras.metrics.MeanAbsoluteError(name="mean_absolute_error")]
    )

    return model


In [29]:
tuner = kt.RandomSearch(
    hypermodel=build,
    objective=kt.Objective("val_mean_absolute_error", direction="min"),
    directory="project",
    project_name="Air_Quality_Index",
    max_trials=5,
    executions_per_trial=1,
    overwrite=True
)


In [30]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.3,
    random_state=0
)

# Scale features for more stable neural-network training
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)


Before tuning, verify that the cleaned and split data contains no NaN or infinite values. NaNs in either the features or target can make the loss become `nan` and cause Keras Tuner trials to fail.


In [33]:
print("NaNs in X_train:", np.isnan(X_train).sum())
print("NaNs in X_test:", np.isnan(X_test).sum())
print("NaNs in y_train:", y_train.isna().sum())
print("NaNs in y_test:", y_test.isna().sum())

print("Infs in X_train:", np.isinf(X_train).sum())
print("Infs in X_test:", np.isinf(X_test).sum())
print("Infs in y_train:", np.isinf(y_train).sum())
print("Infs in y_test:", np.isinf(y_test).sum())


Checking X_train for NaNs: 0
Checking y_train for NaNs: 1
Checking X_test for NaNs: 0
Checking y_test for NaNs: 0

Checking X_train for inf values: 0
Checking y_train for inf values: 0
Checking X_test for inf values: 0
Checking y_test for inf values: 0


The important fix is that invalid rows are removed **before** `train_test_split`, so the training and validation sets cannot contain mismatched or invalid target values.


In [34]:
# Final safety check
assert np.isfinite(X_train).all()
assert np.isfinite(X_test).all()
assert np.isfinite(y_train.to_numpy()).all()
assert np.isfinite(y_test.to_numpy()).all()

print("Data is clean and safe for tuning.")


Checking y_train for NaNs after dropping: 0


In [40]:
tuner.search(
    X_train,
    y_train,
    epochs=20,
    validation_data=(X_test, y_test),
    verbose=1
)

best_hps = tuner.get_best_hyperparameters(num_trials=1)[0]

print("Best number of layers:", best_hps.get("num_layers"))
print("Best learning rate:", best_hps.get("learning_rate"))

for i in range(best_hps.get("num_layers")):
    print(f"Best units in layer {i + 1}:", best_hps.get(f"units_{i}"))


Epoch 1/5
24/24 ━━━━━━━━━━━━━━━━━━━━ 8s 39ms/step - loss: 182.4228 - mean_absolute_error: 182.4228 - val_loss: 63.1839 - val_mean_absolute_error: 63.1839
Epoch 2/5
24/24 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - loss: 87.5324 - mean_absolute_error: 87.5324 - val_loss: 71.4263 - val_mean_absolute_error: 71.4263
Epoch 3/5
24/24 ━━━━━━━━━━━━━━━━━━━━ 1s 24ms/step - loss: 69.1693 - mean_absolute_error: 69.1693 - val_loss: 59.4708 - val_mean_absolute_error: 59.4708
Epoch 4/5
24/24 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - loss: 64.7977 - mean_absolute_error: 64.7977 - val_loss: 67.6161 - val_mean_absolute_error: 67.6161
Epoch 5/5
24/24 ━━━━━━━━━━━━━━━━━━━━ 1s 23ms/step - loss: 70.9328 - mean_absolute_error: 70.9328 - val_loss: 55.3651 - val_mean_absolute_error: 55.3651
Epoch 1/5
24/24 ━━━━━━━━━━━━━━━━━━━━ 8s 52ms/step - loss: 139.6720 - mean_absolute_error: 139.6720 - val_loss: 103.1882 - val_mean_absolute_error: 103.1882
Epoch 2/5
24/24 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - loss: 112.4200 - mean_absolute

/usr/local/lib/python3.12/dist-packages/keras_tuner/src/engine/metrics_tracking.py:111: RuntimeWarning: All-NaN axis encountered
  np.nanmin(values) if self.direction == "min" else np.nanmax(values)


RuntimeError: Number of consecutive failures exceeded the limit of 3.


## Train and evaluate the best model

In [ ]:
best_model = tuner.hypermodel.build(best_hps)

history = best_model.fit(
    X_train,
    y_train,
    epochs=20,
    validation_data=(X_test, y_test),
    verbose=1
)

test_loss, test_mae = best_model.evaluate(X_test, y_test, verbose=0)

print("Test MAE:", test_mae)
